# ThermoGuard Motor — Use the Model
Run the cells in order. Upload `thermoguard_model.pt` when prompted, then move the sliders. The dataset is not required.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from IPython.display import display

INPUTS = ['i_d', 'i_q', 'u_d', 'u_q', 'motor_speed', 'coolant', 'ambient']
TARGETS = ['pm', 'stator_winding', 'stator_tooth', 'stator_yoke']
NAMES = ['Rotor magnet', 'Stator winding', 'Stator tooth', 'Stator yoke']
DT = 2.0
MODEL_PATH = Path('thermoguard_model.pt')
torch.set_num_threads(2)

pd.options.display.float_format = "{:.2f}".format


## 1. Load the trained model

In [2]:
class MotorNet(nn.Module):
    """49 clues, two hidden layers, and four temperature-rise answers."""
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(49, 96), nn.ReLU(),
                                    nn.Linear(96, 64), nn.ReLU(), nn.Linear(64, 4))

    def forward(self, x):
        return self.layers(x)

if not MODEL_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    model_files = [name for name in uploaded if name.endswith('.pt')]
    if len(model_files) != 1:
        raise ValueError('Upload one thermoguard_model.pt file.')
    MODEL_PATH = Path(model_files[0])

saved = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
assert saved['inputs'] == INPUTS and saved['targets'] == TARGETS
assert saved['version'] == 'thermoguard-colab-v1'
net = MotorNet()
net.load_state_dict(saved['weights'])
net.eval()
fallback = np.array(saved['fallback'], dtype=np.float32)
mean = np.array(saved['mean'], dtype=np.float32)
scale = np.array(saved['scale'], dtype=np.float32)
print('Loaded:', MODEL_PATH.name)


Loaded: thermoguard_model.pt


## 2. Prepare inputs and predict
Use the same input preparation as training. Convert network outputs back to degrees Celsius.

In [3]:
def make_features(readings, fallback):
    """Use only current/past inputs; never use measured target temperatures."""
    x = np.array(readings, dtype=np.float32, copy=True)
    if x.ndim != 2 or x.shape[1] != 7 or len(x) == 0:
        raise ValueError('Expected at least one row of seven operating measurements.')
    if not np.isfinite(x[:, 6]).all():
        raise ValueError('Ambient temperature must remain available.')
    available = np.isfinite(x).astype(np.float32)
    ages = np.zeros_like(x)
    # Missing input: hold the last reading for 20 s, then use a training median.
    for channel in range(7):
        last, age = float(fallback[channel]), 0.0
        for row in range(len(x)):
            if available[row, channel]:
                last, age = float(x[row, channel]), 0.0
            else:
                age += DT
                x[row, channel] = last if age <= 20 else fallback[channel]
            ages[row, channel] = min(age, 900) / 900
    current_squared = x[:, 0] ** 2 + x[:, 1] ** 2
    power_proxy = np.abs(x[:, 0] * x[:, 2] + x[:, 1] * x[:, 3])
    base = np.column_stack([x, current_squared, np.abs(x[:, 4]), power_proxy])
    clues = [base]
    # Heat builds and cools gradually: remember four different lengths of history.
    for seconds in [10, 60, 300, 900]:
        history = pd.DataFrame(base[:, [0, 1, 4, 5, 7, 9]]).ewm(
            alpha=1 - np.exp(-DT / seconds), adjust=False).mean()
        clues.append(history.to_numpy(dtype=np.float32))
    elapsed = np.minimum(np.arange(len(x)) * DT, 3600)[:, None] / 3600
    return np.column_stack(clues + [available, ages, elapsed]).astype(np.float32)

def estimate(readings):
    clues = make_features(readings, fallback)
    scaled = (clues - mean) / scale
    with torch.no_grad():
        rise = net(torch.tensor(scaled, dtype=torch.float32)).numpy()
    return readings[:, 6, None] + 50 * rise


## 3. Enter operating values
Move a slider and release it to update the output. **Assumed history: these readings have stayed constant for 30 minutes.** Slider combinations may differ from measured operating conditions; this view is an estimate, not a motor simulation.

In [4]:
import ipywidgets as widgets
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

labels = ['d-axis current (A)', 'q-axis current (A)', 'd-axis voltage (V)',
          'q-axis voltage (V)', 'Motor speed (rpm)', 'Coolant (C)', 'Ambient (C)']
controls = {}
for j, name in enumerate(INPUTS):
    controls[name] = widgets.FloatSlider(
        value=saved['slider_default'][j], min=saved['slider_min'][j],
        max=saved['slider_max'][j], step=1 if j == 4 else 0.1,
        description=labels[j], continuous_update=False,
        style={'description_width': '150px'}, layout=widgets.Layout(width='560px'))
controls['coolant_missing'] = widgets.Checkbox(
    value=False, description='Coolant sensor missing for the last 120 s',
    layout=widgets.Layout(width='560px'))

def show_temperatures(i_d, i_q, u_d, u_q, motor_speed, coolant, ambient, coolant_missing):
    values = np.array([i_d, i_q, u_d, u_q, motor_speed, coolant, ambient], dtype=np.float32)
    # Slider inputs do not include a measured history: assume 30 min of constant readings.
    readings = np.tile(values, (901, 1))
    if coolant_missing:
        readings[-60:, 5] = np.nan
    temperatures = estimate(readings)[-1]
    display(pd.DataFrame({'Component': NAMES, 'Estimated temperature (C)': temperatures.round(2)}))
    fig, ax = plt.subplots(figsize=(7, 3))
    bars = ax.barh(NAMES, temperatures, color=['#245c73', '#d98132', '#3c8d86', '#7d70a6'])
    ax.bar_label(bars, fmt='%.1f C', padding=4)
    ax.set(xlabel='Estimated temperature (C)', xlim=(min(0, temperatures.min() - 10), max(30, temperatures.max() + 25)))
    plt.tight_layout()
    plt.show()
    return temperatures

result = widgets.interactive_output(show_temperatures, controls)
display(widgets.VBox(list(controls.values())), result)


Output()